In [ ]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("thephilliplin/pokemon-showdown-battles-gen9-randbats")
print("Dataset path:", path)

root = Path(path)
json_paths = sorted([str(p) for p in root.rglob("*.json")])
print("Num JSON files found:", len(json_paths))
print("Example file:", json_paths[0] if json_paths else None)


In [ ]:
import json

def safe_load_json(p: str):
    try:
        with open(p, "r") as f:
            return json.load(f)
    except Exception:
        return None

def ingest_battles_to_examples(
    tracker,
    json_paths,
    max_battles=1000,
    verbose_every=200,
):
    examples = []
    battles_loaded = 0

    for p in json_paths:
        battle = safe_load_json(p)
        if not battle or "turns" not in battle:
            continue

        try:
            # your function that yields move examples for both players
            for ex in iter_turn_examples_both_players(tracker, battle):
                examples.append(ex)
        except Exception:
            # skip broken battle logs
            continue

        battles_loaded += 1
        if verbose_every and battles_loaded % verbose_every == 0:
            print(f"battles_loaded={battles_loaded}  examples={len(examples)}")

        if battles_loaded >= max_battles:
            break

    return examples

tracker = BattleStateTracker(form_change_species={"Palafin"})  # keep your Palafin form logic
examples_both = ingest_battles_to_examples(tracker, json_paths, max_battles=1000, verbose_every=200)

print("Total examples:", len(examples_both))
print("Unique battles:", len(set(e["battle_id"] for e in examples_both)))


In [ ]:
import json
from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List, Tuple, Iterator
from copy import deepcopy

STAT_ORDER = ["atk", "def", "spa", "spd", "spe", "accuracy", "evasion"]


@dataclass
class MonState:
    uid: str
    player: str  # "p1" or "p2"
    species: Optional[str] = None
    hp: Optional[int] = None
    max_hp: Optional[int] = None
    fainted: bool = False
    status: Optional[str] = None  # "brn", "par", "psn", "tox", "slp", "frz"
    boosts: Dict[str, int] = field(default_factory=lambda: {k: 0 for k in STAT_ORDER})

    def hp_frac(self) -> Optional[float]:
        if self.hp is None or self.max_hp in (None, 0):
            return None
        return max(0.0, min(1.0, self.hp / self.max_hp))


@dataclass
class SideState:
    player: str
    active_uid: Optional[str] = None
    slot_uids: List[Optional[str]] = field(default_factory=lambda: [None] * 6)


class BattleStateTracker:
    """
    Tracker tailored to your Gen9 RandBats JSON schema:
      - team_revelation.teams.p1/p2
      - turns[].turn_number
      - events with *_uid fields (pokemon_uid, into_uid, target_uid)

    v0 goal: generate per-turn supervised examples for predicting move_id.
    """

    def __init__(self, form_change_species: Optional[set] = None):
        self.form_change_species = form_change_species or {"Palafin"}
        self.reset()

    def reset(self):
        self.battle_id: Optional[str] = None
        self.turn_index: int = 0
        self.mons: Dict[str, MonState] = {}
        self.sides: Dict[str, SideState] = {"p1": SideState("p1"), "p2": SideState("p2")}
        # helper: species -> "canonical slot" for form-change mapping
        self.species_slot: Dict[Tuple[str, str], int] = {}  # (player, species) -> slot

    # ---------- Loading ----------
    def load_battle(self, battle: Dict[str, Any]):
        self.reset()
        self.battle_id = battle.get("battle_id")

        # Initialize mons from team_revelation
        team_rev = battle.get("team_revelation", {})
        teams = team_rev.get("teams", {})

        for player in ("p1", "p2"):
            roster = teams.get(player, [])
            slot = 0
            for entry in roster:
                uid = entry.get("pokemon_uid")
                species = entry.get("species")
                if not uid:
                    continue

                ms = self._ensure_mon(uid, player)
                ms.species = species

                # base_stats.hp is present (max HP)
                base_stats = entry.get("base_stats", {})
                base_hp = base_stats.get("hp")
                if isinstance(base_hp, int):
                    ms.max_hp = base_hp

                # assign stable slots 0..5 using "first time we see a species/uid" logic
                # NOTE: randbats can show extra UIDs for form changes. We'll map those later.
                if slot < 6 and self.sides[player].slot_uids[slot] is None:
                    self.sides[player].slot_uids[slot] = uid
                    if species and (player, species) not in self.species_slot:
                        self.species_slot[(player, species)] = slot
                    slot += 1

        self.turn_index = 0

    def _ensure_mon(self, uid: str, player: str) -> MonState:
        if uid in self.mons:
            return self.mons[uid]
        ms = MonState(uid=uid, player=player)
        self.mons[uid] = ms
        return ms

    def _player_from_uid(self, uid: str) -> str:
        if str(uid).startswith("p2"):
            return "p2"
        return "p1"

    # ---------- Snapshot ----------
    def snapshot(self) -> Dict[str, Any]:
        return {
            "turn_index": self.turn_index,
            "p1": {"active_uid": self.sides["p1"].active_uid, "slots": list(self.sides["p1"].slot_uids)},
            "p2": {"active_uid": self.sides["p2"].active_uid, "slots": list(self.sides["p2"].slot_uids)},
            "mons": {
                uid: {
                    "uid": ms.uid,
                    "player": ms.player,
                    "species": ms.species,
                    "hp": ms.hp,
                    "max_hp": ms.max_hp,
                    "hp_frac": ms.hp_frac(),
                    "status": ms.status,
                    "fainted": ms.fainted,
                    "boosts": dict(ms.boosts),
                }
                for uid, ms in self.mons.items()
            },
        }

    # ---------- Action extraction ----------
    @staticmethod
    def extract_action_for_player(turn_events: List[Dict[str, Any]], player: str) -> Optional[Tuple[str, str]]:
        """
        Returns ("move", move_id) or ("switch", into_uid) for the *first* decision event by that player this turn.
        """
        for ev in turn_events:
            if ev.get("player") != player:
                continue
            t = ev.get("type")
            if t == "move":
                move_id = ev.get("move_id")
                if move_id:
                    return ("move", move_id)
            if t == "switch":
                into_uid = ev.get("into_uid")
                if into_uid:
                    return ("switch", into_uid)
        return None

    # ---------- Event application ----------
    def apply_turn(self, turn_obj: Dict[str, Any]):
        self.turn_index = int(turn_obj.get("turn_number", self.turn_index + 1))
        for ev in turn_obj.get("events", []):
            et = ev.get("type")
            if et == "switch":
                self._apply_switch(ev)
            elif et == "damage":
                self._apply_hp_change(ev, kind="damage")
            elif et == "heal":
                self._apply_hp_change(ev, kind="heal")
            elif et == "status_start":
                self._apply_status_start(ev)
            elif et == "stat_change":
                self._apply_stat_change(ev)
            elif et == "faint":
                self._apply_faint(ev)

    def _apply_switch(self, ev: Dict[str, Any]):
        player = ev.get("player")
        uid = ev.get("into_uid") or ev.get("pokemon_uid")
        if player not in ("p1", "p2") or not uid:
            return

        ms = self._ensure_mon(uid, player)

        # Form change handling: if this uid's species matches a known form-change species
        # and we already have a canonical slot for that species, remap the slot to this new uid.
        if ms.species in self.form_change_species and (player, ms.species) in self.species_slot:
            slot = self.species_slot[(player, ms.species)]
            self.sides[player].slot_uids[slot] = uid
        else:
            # If uid isn't in slots yet, put it in the first free slot (rare)
            if uid not in self.sides[player].slot_uids:
                for i in range(6):
                    if self.sides[player].slot_uids[i] is None:
                        self.sides[player].slot_uids[i] = uid
                        if ms.species and (player, ms.species) not in self.species_slot:
                            self.species_slot[(player, ms.species)] = i
                        break

        self.sides[player].active_uid = uid

    def _apply_hp_change(self, ev: Dict[str, Any], kind: str):
        target = ev.get("target_uid")
        if not target:
            return
        ms = self.mons.get(target)
        if ms is None:
            player = self._player_from_uid(target)
            ms = self._ensure_mon(target, player)

        hp_after = ev.get("hp_after")
        max_hp = ev.get("max_hp")

        if hp_after is not None:
            ms.hp = int(hp_after)
        if max_hp is not None:
            ms.max_hp = int(max_hp)

    def _apply_status_start(self, ev: Dict[str, Any]):
        target = ev.get("target_uid")
        status = ev.get("status")
        if not target or not status:
            return
        ms = self.mons.get(target)
        if ms is None:
            player = self._player_from_uid(target)
            ms = self._ensure_mon(target, player)
        ms.status = status

    def _apply_stat_change(self, ev: Dict[str, Any]):
        target = ev.get("target_uid")
        stat = ev.get("stat")
        amount = ev.get("amount")
        if not target or not stat or amount is None:
            return
        ms = self.mons.get(target)
        if ms is None:
            player = self._player_from_uid(target)
            ms = self._ensure_mon(target, player)

        if stat in ms.boosts:
            ms.boosts[stat] = max(-6, min(6, ms.boosts[stat] + int(amount)))

    def _apply_faint(self, ev: Dict[str, Any]):
        target = ev.get("target_uid")
        if not target:
            return
        ms = self.mons.get(target)
        if ms is None:
            player = self._player_from_uid(target)
            ms = self._ensure_mon(target, player)
        ms.fainted = True
        ms.hp = 0

    # ---------- Dataset generation ----------
    def iter_turn_examples(
        self,
        battle: Dict[str, Any],
        player: str = "p1",
        include_switches: bool = False,
    ) -> Iterator[Dict[str, Any]]:
        """
        Yields per-turn examples:
          {
            "battle_id": str,
            "turn_number": int,
            "state": snapshot_before,
            "action": ("move", move_id) or ("switch", into_uid),
          }
        """
        self.load_battle(battle)

        for turn in battle.get("turns", []):
            events = turn.get("events", [])
            state_before = self.snapshot()

            action = self.extract_action_for_player(events, player)

            # advance state
            self.apply_turn(turn)

            if action is None:
                continue
            if (not include_switches) and action[0] != "move":
                continue

            yield {
                "battle_id": self.battle_id,
                "turn_number": int(turn.get("turn_number")),
                "state": state_before,
                "action": action,
            }


# -----------------------------
# Quick check in your notebook
# -----------------------------
path = "./gen9randombattle-2390494424.json"
with open(path, "r") as f:
    battle = json.load(f)

tracker = BattleStateTracker(form_change_species={"Palafin"})
examples = list(tracker.iter_turn_examples(battle, player="p1", include_switches=False))

print("num examples (p1 move-only):", len(examples))
print("first example:", examples[0]["turn_number"], examples[0]["action"])
print("state_before actives:", examples[0]["state"]["p1"]["active_uid"], examples[0]["state"]["p2"]["active_uid"])
print("mons tracked:", len(examples[0]["state"]["mons"]))


In [ ]:
from typing import List, Dict, Any, Tuple, Optional, Iterator


STATUS_ORDER = ["brn", "par", "psn", "tox", "slp", "frz"]
STAT_ORDER = ["atk", "def", "spa", "spd", "spe", "accuracy", "evasion"]


def one_hot(value: Optional[str], choices: List[str]) -> List[float]:
    return [1.0 if value == c else 0.0 for c in choices]


def safe_hp_frac(mon: Dict[str, Any]) -> Tuple[float, float]:
    """
    Returns (hp_frac_value, hp_known_flag)
    """
    hf = mon.get("hp_frac")
    if hf is None:
        return (0.0, 0.0)
    return (float(hf), 1.0)


def mon_features(mon: Optional[Dict[str, Any]]) -> List[float]:
    """
    Features for an active mon.
    If mon is None -> all zeros with known flags = 0.
    """
    if mon is None:
        # hp_frac, hp_known, fainted, 7 boosts, 6 status
        return [0.0, 0.0, 0.0] + [0.0]*len(STAT_ORDER) + [0.0]*len(STATUS_ORDER)

    hp_frac_val, hp_known = safe_hp_frac(mon)
    fainted = 1.0 if mon.get("fainted") else 0.0

    boosts = mon.get("boosts", {}) or {}
    boost_vec = [float(boosts.get(k, 0)) for k in STAT_ORDER]

    status = mon.get("status")
    status_vec = one_hot(status, STATUS_ORDER)

    return [hp_frac_val, hp_known, fainted] + boost_vec + status_vec


def bench_slot_features(mon: Optional[Dict[str, Any]]) -> List[float]:
    """
    Tiny bench representation: fainted, hp_known, hp_frac.
    """
    if mon is None:
        return [0.0, 0.0, 0.0]
    hp_frac_val, hp_known = safe_hp_frac(mon)
    fainted = 1.0 if mon.get("fainted") else 0.0
    return [fainted, hp_known, hp_frac_val]


def encode_state_v0(state: Dict[str, Any], perspective_player: str) -> List[float]:
    """
    Fixed-length vector from the snapshot "state", from the view of perspective_player.
    """
    other = "p2" if perspective_player == "p1" else "p1"

    mons = state["mons"]
    my_active_uid = state[perspective_player]["active_uid"]
    opp_active_uid = state[other]["active_uid"]

    my_active = mons.get(my_active_uid) if my_active_uid else None
    opp_active = mons.get(opp_active_uid) if opp_active_uid else None

    vec: List[float] = []
    vec += mon_features(my_active)
    vec += mon_features(opp_active)

    # Bench slots (6) for each side based on stable slot list
    my_slots = state[perspective_player]["slots"]
    opp_slots = state[other]["slots"]

    for uid in my_slots:
        vec += bench_slot_features(mons.get(uid) if uid else None)

    for uid in opp_slots:
        # You can make opponent bench even smaller if you want,
        # but keeping same 3 features is fine for v0.
        vec += bench_slot_features(mons.get(uid) if uid else None)

    # Optional: add normalized turn index signal (helps early vs late)
    # Use a simple capped scaling
    t = float(state.get("turn_index", 0))
    vec.append(min(t, 50.0) / 50.0)

    return vec


def iter_turn_examples_both_players(tracker, battle: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
    """
    Yields move-only examples for BOTH p1 and p2.
    Each example has a 'player' field indicating perspective.
    """
    # We run two passes because each pass consumes state progression.
    for player in ("p1", "p2"):
        for ex in tracker.iter_turn_examples(battle, player=player, include_switches=False):
            yield {
                "battle_id": ex["battle_id"],
                "turn_number": ex["turn_number"],
                "player": player,
                "state": ex["state"],
                "action": ex["action"],  # ("move", move_id)
            }


def build_move_vocab(examples: List[Dict[str, Any]], min_count: int = 1) -> Dict[str, int]:
    """
    Move vocab from your dataset. If you later want to cut rare moves, increase min_count.
    """
    counts: Dict[str, int] = {}
    for ex in examples:
        kind, move_id = ex["action"]
        if kind != "move":
            continue
        counts[move_id] = counts.get(move_id, 0) + 1

    # Reserve 0 for UNK if you want to allow unseen moves at inference time
    vocab: Dict[str, int] = {"<UNK>": 0}
    for move_id, c in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
        if c >= min_count:
            vocab[move_id] = len(vocab)
    return vocab


def vectorize_dataset(examples: List[Dict[str, Any]], move_vocab: Dict[str, int]) -> Tuple[List[List[float]], List[int]]:
    X: List[List[float]] = []
    y: List[int] = []
    for ex in examples:
        player = ex["player"]
        vec = encode_state_v0(ex["state"], perspective_player=player)
        _, move_id = ex["action"]
        label = move_vocab.get(move_id, move_vocab["<UNK>"])
        X.append(vec)
        y.append(label)
    return X, y


# -----------------------------
# Run on your single battle file
# -----------------------------
path = "./gen9randombattle-2390494424.json"
with open(path, "r") as f:
    battle = json.load(f)

tracker = BattleStateTracker(form_change_species={"Palafin"})
examples_both = list(iter_turn_examples_both_players(tracker, battle))

print("num examples (both players, move-only):", len(examples_both))
print("sample:", examples_both[0]["player"], examples_both[0]["turn_number"], examples_both[0]["action"])

move_vocab = build_move_vocab(examples_both, min_count=1)
print("num move classes (including <UNK>):", len(move_vocab))
print("top 10 vocab items:", list(move_vocab.items())[:10])

X, y = vectorize_dataset(examples_both, move_vocab)
print("X dim:", len(X), "x", len(X[0]))
print("y dim:", len(y))
print("first y:", y[0])
print(y)


In [ ]:
import numpy as np

# Convert lists -> numpy arrays (this enables fancy indexing)
X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)

print("X type:", type(X), "shape:", X.shape)
print("y type:", type(y), "shape:", y.shape)
print("sample y:", y[:10])


In [ ]:
import numpy as np
import random
from collections import defaultdict

def group_split_by_battle_id(examples, val_ratio=0.2, seed=42):
    by_battle = defaultdict(list)
    for i, ex in enumerate(examples):
        by_battle[ex["battle_id"]].append(i)

    battle_ids = list(by_battle.keys())
    rng = random.Random(seed)
    rng.shuffle(battle_ids)

    n_val = max(1, int(len(battle_ids) * val_ratio))
    val_battles = set(battle_ids[:n_val])

    train_idx, val_idx = [], []
    for bid, idxs in by_battle.items():
        (val_idx if bid in val_battles else train_idx).extend(idxs)

    return np.array(train_idx, dtype=np.int64), np.array(val_idx, dtype=np.int64)

train_idx, val_idx = group_split_by_battle_id(examples, val_ratio=0.2, seed=42)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape)

print("num classes:", len(move_vocab))


In [ ]:
# SETUP MODEL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(move_vocab)
input_dim = X_train.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(num_classes)  # logits
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="top1"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
    ],
)

model.summary()


In [ ]:
# TRAAAAAAAAAAAAAAAAAAAIN 
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_top3", patience=3, mode="max", restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)
